IR Q&A

In [ ]:
import nltk
nltk.download('punkt')

# define a list of documents
print(df['targetParagraphs'][8])
documents = df['targetParagraphs'][8].split('\n')
print(documents)
print('\n')

# preprocess the documents
processed_docs = [doc.lower() for doc in documents]
processed_docs = [nltk.word_tokenize(doc) for doc in processed_docs]

# define a function to answer a question
def answer_question(question, documents):
    # preprocess the question
    processed_question = question.lower()
    processed_question = nltk.word_tokenize(processed_question)
    
    # search for relevant documents
    relevant_docs = []
    for i, doc in enumerate(processed_docs):
        for word in processed_question:
            if word in doc:
                relevant_docs.append(i)
                break
    
    # return the relevant documents
    return [documents[i] for i in relevant_docs]

# example usage
question = df['targetTitle'][8]
answer = answer_question(question, documents)
print(answer)


In [ ]:
import pandas as pd
df = pd.read_csv('valid-output-with-newline.csv')

In [ ]:
import re
import nltk
from nltk.corpus import stopwords
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

ir_predicted = []
for i in range(0, len(df)):
  # Define the corpus
  corpus = df['targetParagraphs'][i].split('\n')

  # Define the function to preprocess the text
  def preprocess(text):
      # Remove non-alphanumeric characters and convert to lowercase
      text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
      text = text.lower()
      # Tokenize the text
      tokens = nltk.word_tokenize(text)
      # Remove stop words
      stop_words = stopwords.words('english')
      tokens = [token for token in tokens if token not in stop_words]
      # Lemmatize the text
      lemmatizer = WordNetLemmatizer()
      tokens = [lemmatizer.lemmatize(token) for token in tokens]
      # Join the tokens back into a string
      text = ' '.join(tokens)
      return text

  # Preprocess the corpus
  preprocessed_corpus = [preprocess(text) for text in corpus]

  # Define the function to get the answer to a question
  def get_answer(question):
      # Preprocess the question
      preprocessed_question = preprocess(question)
      # Vectorize the corpus and the question
      vectorizer = TfidfVectorizer()
      vectorized_corpus = vectorizer.fit_transform(preprocessed_corpus)
      vectorized_question = vectorizer.transform([preprocessed_question])
      # Compute the cosine similarity between the question and each document in the corpus
      similarities = cosine_similarity(vectorized_question, vectorized_corpus)
      # Get the index of the document with the highest similarity score
      index = similarities.argmax()
      # Get the corresponding document
      answer = corpus[index]
      return answer

  # Ask a question and get the answer
  question = str(df['targetTitle'][i])
  answer = get_answer(question)
  print(answer)
  ir_predicted.append(answer)


In [ ]:
df['postText'][8]

In [ ]:
df['targetTitle'][8]

In [ ]:
df['targetParagraphs'][8].split('\n')

In [ ]:
df['prediction'] = ir_predicted

In [ ]:
df.to_csv('find-scores.csv')